# Homework 5

# Задача №1 - Можете ли вы отличить сорняки от рассады?

Теперь приступим к задаче классификации на картинках. Реализуйте программу, которая определяет тип рассады на изображении. 

Для того, чтобы определить характерные особенности каждого типа рассады, у вас есть train. Train это папка, в которой картинки уже классифицированы и лежат в соответствующих папках. Исходя из этой информации можете найти признаки, присущие конкретному растению.

Проверка вашего решения будет на происходить на test. В папке test уже нет метки класса для каждой картинки. 

[Ссылка на Яндекс-диск](https://yadi.sk/d/0Zzp0klXT0iRmA), все картинки тут.

Примеры изображений для теста:
<table><tr>
    <td> <img src="https://i.ibb.co/tbqR37m/fhj.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/6yL3Wmt/sfg.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/pvn7NvF/asd.png" alt="Drawing" style="width: 200px;"/> </td>
</tr></table>

In [6]:
import cv2
import os
import numpy as np
from collections import defaultdict
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

def extract_sift_features(img_path, use_mask=True):
    """Извлекает SIFT-дескрипторы с опциональной маской зелёного цвета"""
    img = cv2.imread(img_path)
    if img is None:
        return None
    
    if use_mask:
        # Создаём маску для зелёных областей (растений)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, (35, 50, 50), (85, 255, 255))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.bitwise_and(gray, gray, mask=mask)
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    sift = cv2.SIFT_create()
    _, descriptors = sift.detectAndCompute(gray, None)
    return descriptors

def train_kmeans(train_folder, n_clusters=100, random_state=42):
    """Обучает K-Means на SIFT-дескрипторах из train"""
    all_descriptors = []
    
    for class_name in os.listdir(train_folder):
        class_path = os.path.join(train_folder, class_name)
        if not os.path.isdir(class_path):
            continue
            
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            descriptors = extract_sift_features(img_path)
            if descriptors is not None:
                all_descriptors.append(descriptors)    
    
    all_descriptors = np.vstack(all_descriptors)
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    kmeans.fit(all_descriptors)
    return kmeans

def image_to_histogram(img_path, kmeans, n_clusters):
    """Преобразует изображение в гистограмму визуальных слов"""
    descriptors = extract_sift_features(img_path)
    
    visual_words = kmeans.predict(descriptors)
    hist, _ = np.histogram(visual_words, bins=n_clusters, range=(0, n_clusters))
    return hist / hist.sum() if hist.sum() > 0 else hist


In [7]:
train_folder = "plants/train"
test_folder = "plants/test"
output_folder = "results"
n_clusters = 100
random_state = 42

print("Обучение K-Means...")
kmeans = train_kmeans(train_folder, n_clusters, random_state)

print("Подготовка данных...")
X_train, y_train = [], []
class_names = sorted(os.listdir(train_folder))

for label, class_name in enumerate(class_names):
    class_path = os.path.join(train_folder, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        hist = image_to_histogram(img_path, kmeans, n_clusters)
        X_train.append(hist)
        y_train.append(label)    

print("Обучение классификатора...")
classifier = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=random_state))
])
classifier.fit(X_train, y_train)

print("Классификация тестовых изображений...")
results = defaultdict(list)

for img_name in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_name)
    hist = image_to_histogram(img_path, kmeans, n_clusters)
    proba = classifier.predict_proba([hist])[0]
    pred_class = np.argmax(proba)
    results[class_names[pred_class]].append(img_name)    

print("Сохранение результатов...")
os.makedirs(output_folder, exist_ok=True)

for class_name, images in results.items():
    class_dir = os.path.join(output_folder, class_name)
    os.makedirs(class_dir, exist_ok=True)
    
    for img_name in images:
        src_path = os.path.join(test_folder, img_name)
        dst_path = os.path.join(class_dir, img_name)
        img = cv2.imread(src_path)
        if img is not None:
            cv2.imwrite(dst_path, img)

print("Результаты сохранены в папку 'results'.")

Обучение K-Means...
Подготовка данных...
Обучение классификатора...
Классификация тестовых изображений...
Сохранение результатов...
Результаты сохранены в папку 'results'.


# Задача №2 - Собери пазл (2.0).

Даны кусочки изображения, ваша задача склеить пазл в исходную картинку. 

Условия:
* Дано исходное изображение для проверки, использовать собранное изображение в самом алгоритме нельзя;
* Картинки имеют друг с другом пересечение;
* После разрезки кусочки пазлов не были повернуты или отражены;
* НЕЛЬЗЯ выбрать опорную картинку для сбора пазла, как это было в homework 3
* В процессе проверки решения пазлы могут быть перемешаны, т.е. порядок пазлов в проверке может отличаться от исходного 

Изображения расположены по [ссылке](https://disk.yandex.ru/d/XtpawH1sV9UDlg).

Примеры изображений:
<img src="puzzle/su_fighter.jpg" alt="Drawing" style="width: 300px;"/>
<table><tr>
    <td> <img src="puzzle/su_fighter_shuffle/0.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/1.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/2.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/3.jpg" alt="Drawing" style="width: 200px;"/> </td>
</tr></table>

In [ ]:
import cv2
import numpy as np
import os
from skimage.metrics import structural_similarity as ssim
from glob import glob
import matplotlib.pyplot as plt

def load_pieces(folder):
    """Загружает все фрагменты пазла из указанной папки"""
    pieces = []
    for filename in sorted(os.listdir(folder)):
        img = cv2.imread(os.path.join(folder, filename))
        if img is not None:
            pieces.append(img)
    return pieces

def apply_transformation(transform_mat, points):
    transformed_points = (transform_mat @ points.T).T
    transformed_points = np.divide(transformed_points.T, transformed_points[:, 2]).T
    return transformed_points

def construct_offset_transformation(x):
    transformation = np.eye(3)
    transformation[:2, 2] = x
    return transformation

def find_offset_ransac(points, transformed_points):
    best_offset = None
    best_loss = np.inf
    loss_f = lambda x1, x2: np.linalg.norm(x1 - x2)
    for point, transformed_point in zip(points, transformed_points):
        offset = (transformed_point - point)[:2]
        transformation = construct_offset_transformation(offset)
        tmp = apply_transformation(transformation, points)
        tmp_loss = loss_f(tmp, transformed_points)
        if tmp_loss < best_loss:
            best_loss = tmp_loss
            best_offset = offset
    return best_offset

def get_shift(first, second):
    nfetures = first.size // second.size * 200

    hyp_params2 = dict(
        nfeatures=200,
        nOctaveLayers=3,
        contrastThreshold=0.03,
        edgeThreshold=10,
        sigma=1.6
    ) 

    hyp_params1 = dict(
        nfeatures=nfetures,
        nOctaveLayers=3,
        contrastThreshold=0.03,
        edgeThreshold=10,
        sigma=1.6
    ) 
    
    sift2 = cv2.SIFT_create(**hyp_params2)
    sift1 = cv2.SIFT_create(**hyp_params1)
    
    FLANN_INDEX_KDTREE = 2
    index_params = dict(algorithm=2, trees=15)
    search_params = dict(checks=150)
    flann = cv2.FlannBasedMatcher(index_params, search_params)

    kp1, des1 = sift1.detectAndCompute(first, None)
    kp2, des2 = sift2.detectAndCompute(second, None)

    if des2 is None or des1 is None:
        return None
    
    try:
        matches = flann.knnMatch(des1, des2, k=2)
        ratio_thresh = 0.4
        good_matches = []
        for m, n in matches:
            if m.distance < ratio_thresh * n.distance:
                good_matches.append(m)
    except:
        return None
        
    if len(good_matches) == 0:
        return None
        
    points = np.array([[kp1[m.queryIdx].pt[1], kp1[m.queryIdx].pt[0], 1] for m in good_matches])
    transformed_points = np.array([[kp2[m.trainIdx].pt[1], kp2[m.trainIdx].pt[0], 1] for m in good_matches])
    
    shift = find_offset_ransac(transformed_points, points)
    if shift is not None:
        shift[0] = round(shift[0])
        shift[1] = round(shift[1])
    return shift

def are_shifted(first, second):
    shift = get_shift(first, second)
    if shift is None:
        return False
    return True

def stitch_images_right_down(left, right, shift):
    new_shape = [max(left.shape[0], right.shape[0] + shift[0]), 
                max(left.shape[1], right.shape[1] + shift[1]), 3]
    new_image = np.zeros(shape=new_shape)
    new_image[0:left.shape[0], 0:left.shape[1], :] = left
    new_image[shift[0]:right.shape[0]+shift[0], shift[1]:right.shape[1]+shift[1], :] = right
    new_image = new_image.astype(np.uint8)
    return new_image

def stitch_images(first, second):
    shift = get_shift(first, second)
    if shift is None:
        return None
    shift = np.array(shift, dtype=np.int32)
    if shift[0] >= 0 and shift[1] >= 0:
        result = stitch_images_right_down(first, second, shift)
    elif shift[0] >= 0 and shift[1] < 0:
        shift[1] *= -1
        first = np.pad(first, ((0, 0), (shift[1], 0), (0, 0)))
        shift[1] = 0
        result = stitch_images_right_down(first, second, shift)
    elif shift[0] < 0 and shift[1] >= 0:
        shift[0] *= -1
        first = np.pad(first, ((shift[0], 0), (0, 0), (0, 0)))
        shift[0] = 0
        result = stitch_images_right_down(first, second, shift)
    elif shift[0] < 0 and shift[1] < 0:
        shift[0] *= -1
        shift[1] *= -1
        first = np.pad(first, ((shift[0], 0), (shift[1], 0), (0, 0)))
        shift[0] = 0
        shift[1] = 0
        result = stitch_images_right_down(first, second, shift)
    return result

def assemble_puzzle_sift(pieces):
    """Собирает пазл из фрагментов с использованием SIFT"""
    if not pieces:
        return None
    
    index = [*range(len(pieces))]
    base_image = pieces[0]
    index.remove(0)

    while len(index) > 0:
        breaked = False
        for i in index:
            img = pieces[i]
            if are_shifted(base_image, img):
                breaked = True
                base_image = stitch_images(base_image, img)
                if base_image is None:
                    continue
                index.remove(i)
                break
        if not breaked:
            print("Не удалось найти подходящий фрагмент для сборки")
            break
            
    return base_image

def save_result(image, output_folder, filename="su_fighter_restored.jpg"):
    """Сохраняет собранное изображение"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    cv2.imwrite(os.path.join(output_folder, filename), image)

# Загрузка и сборка пазлов
pieces1 = load_pieces("puzzle/su_fighter_shuffle")
pieces2 = load_pieces("puzzle/china_shuffle")
pieces3 = load_pieces("puzzle/home_shuffle")

# Сборка с использованием SIFT
result1 = assemble_puzzle_sift(pieces1)
result2 = assemble_puzzle_sift(pieces2)
result3 = assemble_puzzle_sift(pieces3)

# Сохранение результатов
if result1 is not None:
    save_result(result1, "restored_photo", "su_fighter_restored.jpg")
    print("Пазл 1 успешно собран")
else:
    print("Не удалось собрать пазл 1")

if result2 is not None:
    save_result(result2, "restored_photo", "china_restored.jpg")
    print("Пазл 2 успешно собран")
else:
    print("Не удалось собрать пазл 2")

if result3 is not None:
    save_result(result3, "restored_photo", "home.jpg")
    print("Пазл 3 успешно собран")
else:
    print("Не удалось собрать пазл 3")


Пазл 1 успешно собран
Не удалось собрать пазл 2
Не удалось собрать пазл 3
Размер result1: (220, 820, 3)
Размер result2: None
Размер result3: None
